# 00 — Set up PawShield in your workspace

This notebook lands the PawShield dataset in your Unity Catalog. After
it completes (~2 minutes on Databricks default serverless compute),
the following are queryable:

* Seven Delta tables under `${catalog}.pawshield.*`
* 870 file artifacts under `/Volumes/${catalog}/pawshield/bootstrap/`
* Sarah Chen's customer record, two policies, and claim
  `CLM-2026-03-00471`

## Prerequisites
If you follow the book with default settings, a metastore admin must
first create the `genaicert` catalog; this notebook then creates the
schemas and Volume inside it.

## How to run it

Run all cells — this setup notebook is the one place where Run-all is
the intended flow (it provisions everything in order).

The notebook itself fetches the bootstrap data from the companion
GitHub repo (`snowleafbooks/databricks-genai`), creates the schemas
and Volume (the catalog must already exist — see Prerequisites),
stages the data into the Volume, loads the seven
Delta tables, and verifies Sarah's lifecycle anchor. No local CLI, no
manual file upload, no cluster creation — default serverless does the
whole thing.


## 1. Create catalog
![image_1780119786360.png](./image_1780119786360.png "image_1780119786360.png")

In [0]:
## if you use your own catalog, update it here or in the notebook parameter at the top
default_catalog = 'genaicert'
dbutils.widgets.text("catalog", default_catalog)
volume_root = f'/Volumes/{default_catalog}/pawshield/bootstrap'
dbutils.widgets.text("volume_root", volume_root)
catalog = dbutils.widgets.get("catalog")
git_url = 'https://github.com/snowleafbooks/databricks-genai'
print(f"Catalog:     {catalog}")
print(f"Git Repo: {git_url}")

Catalog:     genaicert
Git Repo: https://github.com/snowleafbooks/databricks-genai


## 2. Create the schemas (if absent) and the `bootstrap` Volume

In [0]:
%sql
-- NOTE: create the catalog manually beforehand; this notebook creates only schemas + the Volume
CREATE SCHEMA IF NOT EXISTS identifier(:catalog).pawshield;
CREATE SCHEMA IF NOT EXISTS identifier(:catalog).eval;
CREATE SCHEMA IF NOT EXISTS identifier(:catalog).monitoring;
CREATE VOLUME IF NOT EXISTS identifier(:catalog).pawshield.bootstrap;

## 3. Get all files to local

Clone git data and copy to destination volumes.


In [0]:
!git clone https://github.com/snowleafbooks/databricks-genai.git /tmp/repo

In [0]:
!cp -r /tmp/repo/volume-bootstrap/* /Volumes/{catalog}/pawshield/bootstrap/

In [0]:
!cp -r /tmp/repo/notebooks/* /Workspace/Shared/pawshield-notebooks/

## 4. Load the seven Delta tables from the Volume's parquet seeds

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.pawshield.customers')
  AS SELECT * FROM IDENTIFIER('parquet.`' || :volume_root || '/parquet/customers.parquet`');
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.pawshield.pets')
  AS SELECT * FROM IDENTIFIER('parquet.`' || :volume_root || '/parquet/pets.parquet`');
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.pawshield.policies')
  AS SELECT * FROM IDENTIFIER('parquet.`' || :volume_root || '/parquet/policies.parquet`');
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.pawshield.vets')
  AS SELECT * FROM IDENTIFIER('parquet.`' || :volume_root || '/parquet/vets.parquet`');
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.pawshield.claims')
  AS SELECT * FROM IDENTIFIER('parquet.`' || :volume_root || '/parquet/claims.parquet`');
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.pawshield.claim_line_items')
  AS SELECT * FROM IDENTIFIER('parquet.`' || :volume_root || '/parquet/claim_line_items.parquet`');
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.pawshield.support_tickets')
  AS SELECT * FROM IDENTIFIER('parquet.`' || :volume_root || '/parquet/support_tickets.parquet`');

num_affected_rows,num_inserted_rows


### Row counts (sanity check)

In [0]:
%sql
SELECT 'customers'        AS table_name, count(*) AS row_count FROM identifier(:catalog).pawshield.customers        UNION ALL
SELECT 'pets',                            count(*)              FROM identifier(:catalog).pawshield.pets             UNION ALL
SELECT 'policies',                        count(*)              FROM identifier(:catalog).pawshield.policies         UNION ALL
SELECT 'vets',                            count(*)              FROM identifier(:catalog).pawshield.vets             UNION ALL
SELECT 'claims',                          count(*)              FROM identifier(:catalog).pawshield.claims           UNION ALL
SELECT 'claim_line_items',                count(*)              FROM identifier(:catalog).pawshield.claim_line_items UNION ALL
SELECT 'support_tickets',                 count(*)              FROM identifier(:catalog).pawshield.support_tickets;

table_name,row_count
customers,5000
pets,7414
policies,7414
vets,800
claims,12000
claim_line_items,42055
support_tickets,4500


## 5. Verify the lifecycle anchor: Sarah Chen's case

If any of these assertions fails, stop here and re-run from the top.
The lifecycle thread depends on these exact rows.

In [0]:
# File-existence checks need dbutils.fs.ls — no SQL equivalent for listing
# Volume contents file-by-file.
def _has_file(dirpath: str, filename: str) -> bool:
    try:
        return any(f.name.rstrip("/") == filename for f in dbutils.fs.ls(dirpath))
    except Exception:
        return False

In [0]:
assert _has_file(f"{volume_root}/claim_emails", "sarah_chen_20260328_098473.eml"), \
    "A1 (Sarah's email) missing"
assert _has_file(f"{volume_root}/policy_documents", "pp_plus_tx_v3.pdf"), \
    "A2 (pp_plus_tx_v3.pdf) missing"
assert _has_file(f"{volume_root}/vet_invoices", "austin_vet_00471.pdf"), \
    "A4 (austin_vet_00471.pdf) missing"
print("File artifacts (A1, A2, A4) verified.")

File artifacts (A1, A2, A4) verified.


### A3 — Sarah's claim row
Expected: one row, `amount_billed = 890.00`, `amount_reimbursed = 512.00`,
`status = partial_reimbursement`.

In [0]:
%sql
SELECT claim_id, amount_billed, amount_reimbursed, status
FROM identifier(:catalog).pawshield.claims
WHERE claim_id = 'CLM-2026-03-00471';

claim_id,amount_billed,amount_reimbursed,status
CLM-2026-03-00471,890.0,512.0,partial_reimbursement


## 6. Verify Sarah Chen's customer record resolves

In [0]:
%sql
SELECT customer_id, first_name, last_name,
       address_state, signup_date
FROM identifier(:catalog).pawshield.customers
WHERE customer_id = 'CUST-CHEN-001';

customer_id,first_name,last_name,address_state,signup_date
CUST-CHEN-001,Sarah,Chen,TX,2024-08-12


PawShield is set up. Everything the later notebooks need is in place.